# **Importing the dataset with Data Augmentation**

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import warnings

# Suppress all warnings
warnings.filterwarnings("ignore")


# Define the paths
train_dir = '/content/drive/MyDrive/alzheimer dataset/train'
test_dir = '/content/drive/MyDrive/alzheimer dataset/test'

# Create an ImageDataGenerator object for data augmentation and preprocessing
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.4,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    rotation_range=90,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(rescale=1./255)

# Load the images and labels, ensuring grayscale (1 color channel)
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128, 128),        # Resize images to 128x128
    class_mode='categorical',
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(128, 128),
    class_mode='categorical',

)


2024-08-27 08:45:42.754731: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-27 08:45:42.754848: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-27 08:45:42.935587: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Found 4647 images belonging to 4 classes.
Found 744 images belonging to 4 classes.


# **Transfer Learning**

# **VGG16 Models**

1. VVG16 variant1

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Input, Concatenate, Flatten, Dense, Dropout
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

# 1. Input for grayscale image
input_tensor = Input(shape=(128, 128, 1))

# 2. Convert grayscale → RGB by concatenation
input_tensor_rgb = Concatenate()([input_tensor, input_tensor, input_tensor])
# Now shape becomes (128, 128, 3)

# 3. Load VGG16 WITHOUT top layers
vgg16 = VGG16(
    weights='imagenet',
    include_top=False,
    input_tensor=input_tensor_rgb
)

# 4. Create new model and add VGG as base
modelvgg16 = Sequential()
modelvgg16.add(vgg16)

# 5. Add custom classification head
modelvgg16.add(Flatten())
modelvgg16.add(Dense(256, activation='relu'))
modelvgg16.add(Dropout(0.5))
modelvgg16.add(Dense(128, activation='relu'))
modelvgg16.add(Dropout(0.5))

# Final output for 4 classes
modelvgg16.add(Dense(4, activation='softmax'))

# 6. Freeze VGG16 layers (transfer learning)
for layer in vgg16.layers:
    layer.trainable = False

# 7. Compile model
modelvgg16.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

modelvgg16.summary()


In [ ]:
modelvgg16.fit(train_generator,epochs=50)

Epoch 1/50
 61/146 ━━━━━━━━━━━━━━━━━━━━ 11s 132ms/step - accuracy: 0.4852 - loss: 1.2546

W0000 00:00:1723460619.661058     159 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


146/146 ━━━━━━━━━━━━━━━━━━━━ 21s 105ms/step - accuracy: 0.5133 - loss: 1.1742
Epoch 2/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 13s 86ms/step - accuracy: 0.5650 - loss: 0.9845
Epoch 3/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 13s 86ms/step - accuracy: 0.5779 - loss: 0.9400
Epoch 4/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 14s 90ms/step - accuracy: 0.5784 - loss: 0.9188
Epoch 5/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 13s 88ms/step - accuracy: 0.6138 - loss: 0.8852
Epoch 6/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 13s 86ms/step - accuracy: 0.6204 - loss: 0.8584
Epoch 7/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 13s 85ms/step - accuracy: 0.6166 - loss: 0.8592
Epoch 8/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 13s 87ms/step - accuracy: 0.6278 - loss: 0.8413
Epoch 9/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 14s 92ms/step - accuracy: 0.6346 - loss: 0.8005
Epoch 10/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 14s 90ms/step - accuracy: 0.6365 - loss: 0.7951
Epoch 11/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 13s 86ms/step - accuracy: 0.6333 - loss: 0.7964
Epoch 12/50
146/146 ━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Evaluate the model on the test data
loss, accuracy4 = modelvgg16.evaluate(test_generator)

# Print the accuracy
print(f"Test Accuracy of CNN1: {accuracy4}")


24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 100ms/step - accuracy: 0.7653 - loss: 0.5489
Test Accuracy of CNN1: 0.7715053558349609


2. VVG16 variant2 - custom vgg18 , 2 Conv2D layers

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Dense, Dropout, Conv2D, MaxPooling2D,
                                     Input, GlobalAveragePooling2D, BatchNormalization)
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

# 1-channel → 3-channel
input_tensor = Input(shape=(128,128,1))
input_rgb = tf.keras.layers.Concatenate()([input_tensor, input_tensor, input_tensor])

# Load VGG16 pretrained on ImageNet
vgg16 = VGG16(weights='imagenet', include_top=False, input_tensor=input_rgb)
vgg16.trainable = False    # freeze

modelvgg16_plus_2 = Sequential()
modelvgg16_plus_2.add(vgg16)

# Custom conv layers
modelvgg16_plus_2.add(Conv2D(128, (3,3), padding='same', activation='relu'))
modelvgg16_plus_2.add(BatchNormalization())
modelvgg16_plus_2.add(MaxPooling2D((2,2)))

modelvgg16_plus_2.add(Conv2D(128, (3,3), padding='same', activation='relu'))
modelvgg16_plus_2.add(BatchNormalization())
modelvgg16_plus_2.add(MaxPooling2D((2,2)))

modelvgg16_plus_2.add(Flatten())
# Replace Flatten with GAP
#modelvgg16_plus_2.add(GlobalAveragePooling2D())

# Dense layers
modelvgg16_plus_2.add(Dense(256, activation='relu'))
modelvgg16_plus_2.add(Dropout(0.5))

modelvgg16_plus_2.add(Dense(128, activation='relu'))
modelvgg16_plus_2.add(Dropout(0.5))

# Output layer
modelvgg16_plus_2.add(Dense(4, activation='softmax'))

# Compile
modelvgg16_plus_2.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
modelvgg16_plus_2.fit(train_generator,epochs=50)

Epoch 1/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 24s 108ms/step - accuracy: 0.5094 - loss: 1.1067
Epoch 2/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 14s 88ms/step - accuracy: 0.5678 - loss: 0.9545
Epoch 3/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 14s 91ms/step - accuracy: 0.5919 - loss: 0.9106
Epoch 4/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 14s 89ms/step - accuracy: 0.6124 - loss: 0.8633
Epoch 5/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 13s 86ms/step - accuracy: 0.6240 - loss: 0.8201
Epoch 6/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 14s 92ms/step - accuracy: 0.6480 - loss: 0.7963
Epoch 7/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 14s 90ms/step - accuracy: 0.6612 - loss: 0.7531
Epoch 8/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 14s 90ms/step - accuracy: 0.6598 - loss: 0.7673
Epoch 9/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 14s 88ms/step - accuracy: 0.6769 - loss: 0.7392
Epoch 10/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 13s 87ms/step - accuracy: 0.6882 - loss: 0.7112
Epoch 11/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 14s 89ms/step - accuracy: 0.6954 - loss: 0.6970
Epoch 12/50
146/146 ━━━━━━━━━

In [ ]:

# Evaluate the model on test data
loss, accuracy5 = modelvgg16_plus_2.evaluate(test_generator)
print(f"Test Accuracy: {accuracy5 * 100:.2f}%")


24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 103ms/step - accuracy: 0.9337 - loss: 0.1853
Test Accuracy: 93.55%


# **VGG19 MODELS**

1. VGG19 variant1

In [ ]:
from tensorflow.keras.applications import VGG19
from tensorflow.keras.layers import Input, Concatenate, Flatten, Dense, Dropout
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

# 1. Input for grayscale image
input_tensor = Input(shape=(128, 128, 1))

# 2. Convert grayscale → RGB by stacking 3 channels
input_tensor_rgb = Concatenate()([input_tensor, input_tensor, input_tensor])
# Shape: (128, 128, 3)

# 3. Load VGG19 without top layers
vgg19 = VGG19(
    weights='imagenet',
    include_top=False,
    input_tensor=input_tensor_rgb
)

# 4. Build the model using Sequential
modelvgg19 = Sequential()
modelvgg19.add(vgg19)

# 5. Add custom classifier head
modelvgg19.add(Flatten())
modelvgg19.add(Dense(256, activation='relu'))
modelvgg19.add(Dropout(0.5))
modelvgg19.add(Dense(128, activation='relu'))
modelvgg19.add(Dropout(0.5))

# Output layer for 4 classes
modelvgg19.add(Dense(4, activation='softmax'))

# 6. Freeze VGG19 convolutional layers
for layer in vgg19.layers:
    layer.trainable = False

# 7. Compile model
modelvgg19.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

modelvgg19.summary()


In [ ]:
modelvgg19.fit(train_generator,epochs=50)

Epoch 1/50
 30/146 ━━━━━━━━━━━━━━━━━━━━ 21s 185ms/step - accuracy: 0.4185 - loss: 1.3791

W0000 00:00:1723462051.018908     158 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


146/146 ━━━━━━━━━━━━━━━━━━━━ 21s 109ms/step - accuracy: 0.4969 - loss: 1.1933
Epoch 2/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 13s 86ms/step - accuracy: 0.5693 - loss: 1.0073
Epoch 3/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 14s 93ms/step - accuracy: 0.5950 - loss: 0.9370
Epoch 4/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 14s 91ms/step - accuracy: 0.6063 - loss: 0.9074
Epoch 5/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 15s 97ms/step - accuracy: 0.6031 - loss: 0.8887
Epoch 6/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 13s 87ms/step - accuracy: 0.5966 - loss: 0.8846
Epoch 7/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 13s 86ms/step - accuracy: 0.6039 - loss: 0.8684
Epoch 8/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 13s 87ms/step - accuracy: 0.6018 - loss: 0.8614
Epoch 9/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 16s 102ms/step - accuracy: 0.6235 - loss: 0.8337
Epoch 10/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 21s 135ms/step - accuracy: 0.6165 - loss: 0.8426
Epoch 11/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 19s 123ms/step - accuracy: 0.6140 - loss: 0.8187
Epoch 12/50
146/146 ━━━━━━━━━━━━━━━━━

In [ ]:

# Evaluate the model on test data
loss, accuracy6 = modelvgg19.evaluate(test_generator)
print(f"Test Accuracy: {accuracy6 * 100:.2f}%")


24/24 ━━━━━━━━━━━━━━━━━━━━ 4s 114ms/step - accuracy: 0.7438 - loss: 0.6219
Test Accuracy: 73.52%


2. vgg19 variant 2,  2 Conv2D layers

In [ ]:
from tensorflow.keras.applications import VGG19
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout, Conv2D, MaxPooling2D, Input, Concatenate
from tensorflow.keras.optimizers import Adam

# Adjust the input shape to accept grayscale images
input_tensor = Input(shape=(128, 128, 1))
input_tensor_rgb = Concatenate()([input_tensor, input_tensor, input_tensor])
# Shape is now (128, 128, 3)

# Load the VGG19 model with adjusted input (feature extractor only)
vgg19 = VGG19(weights='imagenet', include_top=False, input_tensor=input_tensor_rgb)

# Optionally freeze the VGG19 base (before compiling)
vgg19.trainable = False
# (Equivalent to looping over layers and setting layer.trainable = False)

# Create a Sequential model and add the VGG19 base
modelvgg19_plus_2 = Sequential()
modelvgg19_plus_2.add(vgg19)

# Add two custom Conv2D + MaxPooling2D layers
modelvgg19_plus_2.add(Conv2D(256, (3, 3), activation='relu', padding='same'))
modelvgg19_plus_2.add(MaxPooling2D(pool_size=(2, 2)))
modelvgg19_plus_2.add(Conv2D(256, (3, 3), activation='relu', padding='same'))
modelvgg19_plus_2.add(MaxPooling2D(pool_size=(2, 2)))

# Flatten the output and add fully connected layers
modelvgg19_plus_2.add(Flatten())
modelvgg19_plus_2.add(Dense(256, activation='relu'))
modelvgg19_plus_2.add(Dropout(0.5))
modelvgg19_plus_2.add(Dense(128, activation='relu'))
modelvgg19_plus_2.add(Dropout(0.5))

# Output layer (adjust units to your number of classes)
modelvgg19_plus_2.add(Dense(4, activation='softmax'))

# Compile the model
modelvgg19_plus_2.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',    # use sparse_categorical_crossentropy if labels are integer-encoded
    metrics=['accuracy']
)


In [ ]:
modelvgg19_plus_2.fit(train_generator,epochs=50)

Epoch 1/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 25s 126ms/step - accuracy: 0.5051 - loss: 1.1301
Epoch 2/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 15s 98ms/step - accuracy: 0.5714 - loss: 0.9780
Epoch 3/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 16s 102ms/step - accuracy: 0.5922 - loss: 0.9072
Epoch 4/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 16s 106ms/step - accuracy: 0.6148 - loss: 0.8632
Epoch 5/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 16s 103ms/step - accuracy: 0.6179 - loss: 0.8315
Epoch 6/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 16s 104ms/step - accuracy: 0.6313 - loss: 0.8209
Epoch 7/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 14s 94ms/step - accuracy: 0.6419 - loss: 0.8043
Epoch 8/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 14s 91ms/step - accuracy: 0.6555 - loss: 0.7689
Epoch 9/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 14s 91ms/step - accuracy: 0.6402 - loss: 0.7835
Epoch 10/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 14s 88ms/step - accuracy: 0.6587 - loss: 0.7531
Epoch 11/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 15s 98ms/step - accuracy: 0.6618 - loss: 0.7529
Epoch 12/50
146/146 ━━━━━

In [ ]:

# Evaluate the model on test data
loss, accuracy7 = modelvgg19_plus_2.evaluate(test_generator)
print(f"Test Accuracy: {accuracy7 * 100:.2f}%")


24/24 ━━━━━━━━━━━━━━━━━━━━ 4s 122ms/step - accuracy: 0.8634 - loss: 0.3778
Test Accuracy: 86.29%
